In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [ ]:
# ⬇️ Параметры подключения к PostgreSQL public.shops 

jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shops" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD")


shops_df = spark.read\
		.format("jdbc")\
		.option("url", jdbc_url)\
		.option("user", db_user)\
		.option("password", db_password)\
		.option("dbtable", table_name)\
		.option("fetchsize", 1000)\
		.option("driver", "org.postgresql.Driver")\
		.load() 


shops_df.show()
shops_df.printSchema()

In [ ]:
# ⬇️ Параметры подключения к PostgreSQLpublic.shop_timezone 


jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shop_timezone" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 


shop_timezone_df = spark.read \
				.format("jdbc") \
				.option("url", jdbc_url) \
				.option("user", db_user) \
				.option("password", db_password) \
				.option("dbtable", table_name) \
				.option("fetchsize", 1000) \
				.option("driver", "org.postgresql.Driver") \
				.load() 
shop_timezone_df.show()
shop_timezone_df.printSchema()

In [28]:
shops_df.createOrReplaceTempView("shops") 
shop_timezone_df.createOrReplaceTempView("shop_timezone")

# 1. Готовим "чистый" справочник зон
# Группируем по plant и берем MAX(time_zone). 
# Так как любая строка 'RUS...' больше, чем пустая строка, MAX вытеснит пустоту.
clean_tz = spark.sql("""
    SELECT 
        plant, 
        MAX(time_zone) as raw_tz
    FROM shop_timezone
    WHERE plant IS NOT NULL
    GROUP BY plant
""")
clean_tz.createOrReplaceTempView("clean_tz")

# 2. Теперь джойним магазины с этим ЧИСТЫМ справочником
df_sql = spark.sql("""
    SELECT 
        s.st_id, 
        s.shop_name,
        CASE 
            WHEN c.raw_tz IS NULL OR c.raw_tz = '' THEN '3'
            ELSE CAST(substr(c.raw_tz, 4) AS INT)
        END AS tz_code
    FROM shops s
    LEFT JOIN clean_tz c ON s.st_id = CAST(c.plant AS INT)
    ORDER BY st_id 
""")

df_sql.show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  846|      Lenta|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
|  849|   FixPrice|      3|
|  850|     Magnit|      3|
|  851|      Lenta|      3|
+-----+-----------+-------+



In [29]:
spark.stop()